In [1]:
import numpy as np
import mne
import scipy.io
import matplotlib.pyplot as plt

# 思路：从MAT文件提取EEG，统一为 (Time, Channels)，仅保留前64导并完成滤波、重参考与分段。

def _extract_eeg_matrix(mat):
    eeg = None

    if 'data' in mat:
        data_block = mat['data']
        if isinstance(data_block, np.ndarray) and data_block.size > 0:
            candidate = data_block[0, 0]
            if hasattr(candidate, 'eeg'):
                eeg = np.asarray(candidate.eeg)
            elif isinstance(candidate, dict) and 'eeg' in candidate:
                eeg = np.asarray(candidate['eeg'])
            elif data_block.dtype.names and 'eeg' in data_block.dtype.names:
                eeg = np.asarray(data_block['eeg'][0, 0])
        elif isinstance(data_block, dict) and 'eeg' in data_block:
            eeg = np.asarray(data_block['eeg'])

    if eeg is None and 'eeg' in mat:
        eeg = np.asarray(mat['eeg'])

    if eeg is None:
        raise KeyError("未在MAT文件中找到EEG数据，请检查键名（如 data.eeg 或 eeg）。")

    eeg = np.squeeze(eeg)
    if eeg.ndim != 2:
        raise ValueError(f"EEG数据应为二维矩阵，当前维度: {eeg.ndim}")

    # 统一到 (Time, Channels)
    if eeg.shape[0] <= 128 and eeg.shape[1] > eeg.shape[0]:
        eeg = eeg.T

    if eeg.shape[1] < 64:
        raise ValueError(f"通道数不足64，当前通道数: {eeg.shape[1]}")

    return eeg.astype(np.float32)

def run_task1_preprocessing(file_path, sfreq=512.0, segment_len=512, vis_sec=10):
    mat = scipy.io.loadmat(file_path, squeeze_me=False, struct_as_record=False)
    eeg_time_channel = _extract_eeg_matrix(mat)

    eeg_64 = eeg_time_channel[:, :64].T  # (Channels, Time)
    ch_names = [f'EEG{i+1:03d}' for i in range(64)]
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(eeg_64, info)

    raw_before = raw.copy()
    raw.filter(l_freq=4.0, h_freq=40.0, fir_design='firwin', verbose='ERROR')
    raw.notch_filter(freqs=50.0, verbose='ERROR')

    start_idx = int(vis_sec * sfreq)
    if start_idx >= raw.n_times:
        start_idx = 0
    stop_idx = min(start_idx + int(sfreq), raw.n_times)
    picks = np.random.choice(64, size=10, replace=False)

    data_before, times = raw_before.get_data(
        picks=picks, start=start_idx, stop=stop_idx, return_times=True
    )
    data_after = raw.get_data(picks=picks, start=start_idx, stop=stop_idx)

    plt.figure(figsize=(12, 6))
    plt.subplot(2, 1, 1)
    plt.plot(times, data_before.T)
    plt.title('Before Filtering (1s Segment)')
    plt.subplot(2, 1, 2)
    plt.plot(times, data_after.T)
    plt.title('After 4-40Hz Band-pass + 50Hz Notch')
    plt.tight_layout()
    plt.show()

    raw.set_eeg_reference('average', projection=False)
    final_data = raw.get_data()  # (64, Total_Samples)

    n_total = final_data.shape[1]
    n_segments = n_total // segment_len
    if n_segments == 0:
        raise ValueError(f"数据长度不足一个分段，segment_len={segment_len}, total={n_total}")

    eeg_reshaped = final_data[:, : n_segments * segment_len]
    eeg_reshaped = eeg_reshaped.reshape(64, n_segments, segment_len).transpose(1, 0, 2)
    eeg_reshaped = eeg_reshaped.astype(np.float32)  # (N, 64, 512)

    return eeg_reshaped, raw.info

In [2]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# 思路：
# 1. 将 (N, 64, 512) 展平为 (N, 32768)
# 2. 分层划分训练与测试集
# 3. 对比线性核与RBF核性能

def run_task2_svm(X, y, test_size=0.2, random_state=42):
    if X.ndim != 3:
        raise ValueError(f"X应为三维 (N, C, T)，当前维度: {X.ndim}")
    if len(X) != len(y):
        raise ValueError(f"样本与标签数量不一致: len(X)={len(X)}, len(y)={len(y)}")

    n_samples = X.shape[0]
    X_flat = X.reshape(n_samples, -1)

    stratify_target = y if len(np.unique(y)) > 1 else None
    X_train, X_test, y_train, y_test = train_test_split(
        X_flat, y, test_size=test_size, random_state=random_state, stratify=stratify_target
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    clf_lin = SVC(kernel='linear', C=1.0)
    clf_lin.fit(X_train, y_train)
    acc_lin = accuracy_score(y_test, clf_lin.predict(X_test))

    clf_rbf = SVC(kernel='rbf', C=1.0, gamma='scale')
    clf_rbf.fit(X_train, y_train)
    acc_rbf = accuracy_score(y_test, clf_rbf.predict(X_test))

    print(f"Linear SVM Accuracy: {acc_lin:.4f}")
    print(f"RBF SVM Accuracy: {acc_rbf:.4f}")
    return acc_lin, acc_rbf

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, random_split
from train_utils import get_device, get_dataloaders, train_and_eval_visualized

# 思路：
# 1. 使用train_utils中的真实训练流程
# 2. 将每个样本整理为 (T, C) 以匹配 train_and_eval_visualized 内部 transpose 逻辑
# 3. 使用 ST_AADNet 进行二分类训练

class EEGSegmentDataset(Dataset):
    def __init__(self, X, y):
        if X.ndim != 3:
            raise ValueError(f"X应为三维 (N, C, T)，当前维度: {X.ndim}")
        if len(X) != len(y):
            raise ValueError(f"样本与标签数量不一致: len(X)={len(X)}, len(y)={len(y)}")

        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        # 返回 (T, C)，使 DataLoader batch 后为 (B, T, C)
        frame = self.X[idx].transpose(0, 1)
        label = self.y[idx]
        return frame, label

def run_task3_train_st_aadnet(
    X,
    y,
    epochs=20,
    batch_size=32,
    test_ratio=0.2,
    lr=1e-3,
    save_dir='./checkpoints/st_aadnet'
    ):
    device = get_device()

    dataset = EEGSegmentDataset(X, y)
    n_total = len(dataset)
    n_test = max(1, int(n_total * test_ratio))
    n_train = n_total - n_test
    if n_train <= 0:
        raise ValueError("训练集样本数为0，请减小 test_ratio 或增加样本数")

    generator = torch.Generator().manual_seed(42)
    train_set, test_set = random_split(dataset, [n_train, n_test], generator=generator)

    pin_memory = device.type == 'cuda'
    train_loader, test_loader = get_dataloaders(
        dataset=train_set,
        test_set=test_set,
        batch_size=batch_size,
        num_workers=0,
        pin_memory=pin_memory
    )

    model_cls = globals().get('ST_AADNet')
    if model_cls is None:
        raise NameError("请先运行第4格，定义 ST_AADNet 模型")

    model = model_cls(num_channels=X.shape[1]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    train_and_eval_visualized(
        model=model,
        train_loader=train_loader,
        test_loader=test_loader,
        optimizer=optimizer,
        criterion=criterion,
        epochs=epochs,
        save_dir=save_dir,
        device=device,
        model_name='ST_AADNet'
    )

    return model

In [4]:
import torch
import torch.nn as nn

class ST_AADNet(nn.Module):
    def __init__(self, num_channels=64):
        super().__init__()
        self.num_channels = num_channels

        # 1. 时间卷积: 学习不同频段特征
        self.temp_conv = nn.Conv2d(1, 40, kernel_size=(1, 25), stride=(1, 1))
        # 2. 空间卷积: 跨通道整合信息
        self.spat_conv = nn.Conv2d(40, 30, kernel_size=(num_channels, 1))
        self.bn = nn.BatchNorm2d(30)
        self.pool = nn.AvgPool2d(kernel_size=(1, 20))

        # 3. LSTM层: 捕捉长程动态
        self.lstm = nn.LSTM(input_size=30, hidden_size=30, batch_first=True)
        self.fc = nn.Linear(30, 2)

    def _to_bct(self, x):
        # 目标统一为 (B, C, T)
        if x.dim() != 3:
            raise ValueError(f"输入应为3维张量，当前形状: {tuple(x.shape)}")

        if x.shape[1] == self.num_channels:
            # (B, C, T)
            return x.contiguous()

        if x.shape[2] == self.num_channels:
            if x.shape[0] > x.shape[1]:
                # (T, B, C) -> (B, C, T)
                return x.permute(1, 2, 0).contiguous()
            # (B, T, C) -> (B, C, T)
            return x.permute(0, 2, 1).contiguous()

        raise ValueError(f"无法识别输入维度语义，形状: {tuple(x.shape)}")

    def forward(self, x):
        x = self._to_bct(x)
        x = x.unsqueeze(1)                     # (B, 1, C, T)
        x = torch.square(self.temp_conv(x))    # 非线性能量激活
        x = self.spat_conv(x)
        x = self.bn(x)
        x = self.pool(x)

        x = x.squeeze(2).permute(0, 2, 1)      # (B, Time_Steps, Features)
        _, (hn, _) = self.lstm(x)
        return self.fc(hn.squeeze(0))